# LA Permit Timeline Predictor: New Commercial and Multifamily Buildings

**What this project does.** A developer who files plans for a new apartment, office, retail, industrial, hotel or mixed-use building in Los Angeles wants to know one thing: when will the city issue the building permit? This notebook uses every new commercial and apartment building application filed with the Los Angeles Department of Building and Safety (LADBS) since 2010 to answer that question as honestly as the data allows.

We first try to predict a single number, the days from filing to permit, with a linear regression and a neural network, the same way we predicted student scores in our MAIA lessons. Then we build a stronger model that gives the odds of having a permit within 12, 24 and 36 months. Along the way we show a trap hidden in the city's data that makes a model look far more accurate than it is, and we finish with an input form where you can describe a project and get its odds, a date range and similar past projects.

To run everything: Runtime, then Run all. It takes a few minutes and needs no logins.

# Loading our Data

The applications come straight from the City of Los Angeles open data portal: "Building Permits Submitted" for 2010 to 2019 and for 2020 to today. We keep new buildings only, in the Commercial and Apartment categories. If the city's server does not answer, the notebook falls back to a saved copy of the same data.

Two small helper tables are loaded with it: the permit history of each building site, and reference tables for planning approvals and construction time.

In [ ]:
import io
import re
import time
import urllib.parse
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import ipywidgets as widgets
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

In [ ]:
CITY_DATA = "https://data.lacity.org/resource/{}.csv"
SNAPSHOT = "https://raw.githubusercontent.com/mjingram123/permit-predictor/main/data/"
NEW_BUILDINGS = "permit_type='Bldg-New' AND permit_sub_type in ('Commercial','Apartment')"


def fetch_text(url, attempts=3, timeout=60):
    for attempt in range(attempts):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=timeout) as response:
                return response.read().decode("utf-8")
        except Exception:
            time.sleep(3 * (attempt + 1))
    return None


def load_permits(dataset_id, years, snapshot_name, fewest_rows):
    query = urllib.parse.urlencode({"$where": NEW_BUILDINGS, "$limit": 50000})
    text = fetch_text(CITY_DATA.format(dataset_id) + "?" + query)
    if text is not None:
        table = pd.read_csv(io.StringIO(text), dtype=str)
        if len(table) >= fewest_rows and "submitted_date" in table.columns:
            print(f"Filed {years}: {len(table):,} applications from the live city data")
            return table
    table = pd.read_csv(SNAPSHOT + snapshot_name, dtype=str)
    print(f"Filed {years}: {len(table):,} applications from the saved snapshot")
    return table


def load_helper(name):
    return pd.read_csv(SNAPSHOT + name, dtype={"permit_nbr": str})

In [ ]:
older = load_permits("n3xg-rixm", "2010 to 2019", "permits_2010_2019.csv.gz", 8000)
newer = load_permits("gwh9-jnip", "2020 to today", "permits_2020_present.csv.gz", 5000)
permits = pd.concat([older, newer], ignore_index=True)

site_history = load_helper("parcel_history.csv")
entitlement_times = load_helper("entitlement_lookup.csv")
construction_times = load_helper("construction_lookup.csv").rename(columns={"asset_class": "building_type"})
city_benchmark = load_helper("city_benchmark.csv")

permits[["permit_nbr", "primary_address", "permit_sub_type", "submitted_date", "issue_date", "valuation", "work_desc"]].head()

# Data Cleaning

The raw table mixes real buildings with things we do not want:

- **Revisions** of an earlier application. Their permit number has a middle part that does not end in 0000.
- **Plan-check-only filings.** Developers can ask the city to review zoning early. These filings are never meant to become a permit, so counting them as "never issued" would be wrong.
- **Small structures** filed as new buildings: carports, canopies, signs, trailers, guard booths and so on.

We also sort each real building into a building type. We read the type from the applicant's own project description, not from the city's "use" column, for a reason shown in the next section.

In [ ]:
permits["filed"] = pd.to_datetime(permits.submitted_date, errors="coerce")
permits["permit_date"] = pd.to_datetime(permits.issue_date, errors="coerce")
permits["occupancy_date"] = pd.to_datetime(permits.cofo_date, errors="coerce")
for column in ["valuation", "lat", "lon", "square_footage", "height"]:
    permits[column] = pd.to_numeric(permits[column], errors="coerce")
permits["text"] = permits.work_desc.fillna("").str.upper()
permits = permits[permits.filed.notna()].copy()
DATA_DATE = pd.to_datetime(permits.refresh_time, errors="coerce").max().normalize()

PLAN_CHECK_ONLY = r"ZONING ONLY|ZONING REVIEW ONLY|ZONING PLAN CHECK ONLY|PRELIMINARY ZONING|PDPP|HSAP|PC ONLY|PARALLEL DESIGN|SB ?330"

REAL_BUILDING = r"\d+[- ]?STOR(?:Y|IES).{0,60}(?:UNIT|APARTMENT|OFFICE|RETAIL|HOTEL|WAREHOUSE)"
SMALL_STRUCTURE = (
    r"CARPORT|PATIO|CANOPY|CANOPIES|TRASH ENCLOSURE|\bSIGN\b(?! (?:IS|UNDER|TO BE|PERMIT))|FENCE|RETAINING WALL|\bPOOL\b|\bTENT\b"
    r"|\bTANK\b|ANTENNA|MONOPOLE|SHADE STRUCTURE|SHADE SAIL|GAZEBO|TRELLIS|\bSHED\b|\bBARN\b|GUARD (?:SHACK|BOOTH|HOUSE)|\bBOOTH\b"
    r"|KIOSK|MODULAR|PAY STATION|TEMPORARY|TRAILER|PERGOLA|CABANA|BLEACHER|GENERATOR|EQUIPMENT (?:ENCLOSURE|SHELTER|PAD)"
    r"|RESTROOM BUILDING|PUBLIC RESTROOM|\bNEW RESTROOM\b|TOILET ROOM|PREFAB|BUNGALOW|PORTABLE|RELOCATABLE|CABIN|PUMP HOUSE"
    r"|PUMP ROOM|DUGOUT|PLAY ?GROUND|LUNCH SHELTER|COVERED PARKING|BUS SHELTER|BIKE SHELTER|\bVAULT\b|OPEN STAGE"
    r"|(?<!COIN )LAUNDRY ROOM|MAILBOX|COMMERCIAL\s+COACH|\bCOACH\b.{0,20}(?:OFFICE|CLASSROOM)|HCD APPROVED"
    r"|STATE[- ]APPROVED\s+(?:COMMERCIAL\s+|MODULAR\s+)?(?:COACH|BOOTH|STRUCTURE)"
    r"|STORAGE (?:BLDG|BUILDING).{0,30}ACCESSORY TO|ACCESSORY TO.{0,40}STORAGE (?:BLDG|BUILDING)|PRE[- ]?FAB\w* STORAGE"
)
HOUSE = r"\bADU\b|ACCESSORY DWELLING|\bDUPLEX\b|SINGLE[- ]FAMILY (?:DWELLING|HOUSE|RESIDENCE)|\bTWO[- ]UNIT\b|\b2[- ]UNIT\b|ACCESSORY LAUNDRY"
NOT_A_HOUSE = r"\b(?:[3-9]|\d{2,})[- ]?UNITS?\b|THREE[- ]UNIT|\bAPARTMENT\b|MIX|RETAIL|RESTAURANT|COMMERCIAL|OFFICE|TOWNHO"
SMALL_GARAGE = r"\b(?:\d+|TWO|THREE|FOUR)\s*-?\s*CAR\s+GARAGE"
NOT_A_SMALL_GARAGE = r"UNIT|APART|APT|OFFICE|RETAIL|CONDO|TOWNHO|DWELLING|REPAIR|DEALER"
TINY_FOOTPRINT = r"\b(?:1?\d|2[0-5])'[-\d \"/.]*(?:\"|'')?\s*X\s*(?:1?\d|2[0-5])'"
NOT_TINY = r"UNIT|APART|HOTEL|RETAIL|RESTAURANT|\b[2-9]\d?[- ]STOR"


def is_small_structure(text):
    if re.search(REAL_BUILDING, text):
        return False
    if re.search(SMALL_STRUCTURE, text):
        return True
    if re.search(HOUSE, text) and not re.search(NOT_A_HOUSE, text):
        return True
    if re.search(SMALL_GARAGE, text) and not re.search(NOT_A_SMALL_GARAGE, text):
        return True
    return bool(re.search(TINY_FOOTPRINT, text)) and not re.search(NOT_TINY, text)


MIXED_USE = (
    r"MIX(?:ED)?[-/ ]?USED?|LIVE.?WORK"
    r"|(?:APARTMENT|\d+[- ]UNITS? (?:APARTMENT|AFFORDABLE|SENIOR|SUPPORTIVE)|DWELLING UNITS|RESIDENTIAL UNITS).{0,120}(?:RETAIL|COMMERCIAL|RESTAURANT|MEDICAL OFFICE)"
    r"|(?:RETAIL|COMMERCIAL ?(?:SPACE|USE)|RESTAURANT|MEDICAL OFFICE).{0,60}(?:APARTMENT|\d+[- ]UNITS? (?:APARTMENT|RESIDENTIAL|AFFORDABLE))"
    r"|APT/ ?OFFICE|APARTMENT/ ?OFFICE"
)
CARE_HOME = r"ELDER|ASSISTED|CARE FACILITY|TRANSITIONAL LIVING"
NOT_A_HOTEL = r"RESIDENTIAL HOTEL|(?:ADJOINING|ADJACENT TO|ACCESSORY TO|FOR)\s+(?:THE\s+)?(?:EXISTING|\(E\))\s+HOTEL|\bAPARTMENT (?:BUILDING|COMPLEX)\b"
APARTMENTS = r"\d+[- ]UNITS?\s+(?:\w+\s+){0,5}(?:APARTMENT|APT|CONDO)|APARTMENT (?:BUILDING|BLDG|COMPLEX|HOUSE)|\bAPT\b"
CIVIC = (
    r"SCHOOL|CLASSROOM|ACADEMIC|COLLEGE|UNIVERSIT|\bCAMPUS\b|MUSEUM|CHURCH|MOSQUE|SYNAGOGUE|MAUSOLEUM|COLUMBAR|CRYPT|STADIUM|DORM"
    r"|FITNESS CENTER|RECREATION CENTER|COMMUNITY CENTER|PUBLIC LIBRARY|FIRE STATION"
)
PARKING_BUILDING = r"PARKING (?:STRUCTURE|GARAGE|TIER)"
CLEARLY_RETAIL = r"AUTO (?:REPAIR|SERVICE)|SERVICE SHOP|PHARMACY|PARLOR|CONVENIENCE|MINI.?MART|GAS(?:OLINE)? STATION|(?<!/)RETAIL (?:BLDG|BUILDING|SALES|STORE)|COFFEE|BAKERY|SUPERMARKET|GROCERY"
INDUSTRIAL = r"WAREHOUSE|MANUFACTUR|INDUSTRIAL|TILT|DISTRIBUTION|FABRICAT\w* (?:BLDG|BUILDING|SHOP)|HAZARDOUS|SELF[- ]STORAGE|STORAGE (?:BLDG|BUILDING|FACILITY)"
OFFICE = r"OFFICE|MEDICAL|CLINIC|BANK"
RETAIL = r"RETAIL|RESTAURANT|MARKET|STORE|SHOP|CAR WASH|GAS STATION|DRIVE.?THR|DEALERSHIP|CAFE|COMMERCIAL BUILDING|\bBAR\b|FOOD"


def building_type(text, sub_type):
    if re.search(MIXED_USE, text):
        return "mixed-use"
    if re.search(CARE_HOME, text):
        return "other commercial"
    if re.search(r"HOTEL|MOTEL|GUEST ?ROOMS", text) and not re.search(NOT_A_HOTEL, text):
        return "hotel"
    if sub_type == "Apartment" or re.search(APARTMENTS, text):
        return "multifamily"
    if re.search(CIVIC, text) and not re.search(r"\bWAREHOUSE\b|SELF[- ]STORAGE", text):
        return "other commercial"
    if re.search(PARKING_BUILDING, text) and not re.search(r"OFFICE|RETAIL|MEDICAL|WAREHOUSE|RESTAURANT", text):
        return "other commercial"
    if re.search(CLEARLY_RETAIL, text):
        return "retail"
    if re.search(INDUSTRIAL, text):
        return "industrial"
    if re.search(OFFICE, text):
        return "office"
    if re.search(RETAIL, text):
        return "retail"
    return "other commercial"


permits["is_revision"] = permits.permit_nbr.str.split("-").str[1].str[-4:] != "0000"
permits["is_plan_check_only"] = permits.text.str.contains(PLAN_CHECK_ONLY, regex=True)
permits["is_small_structure"] = permits.text.map(is_small_structure)
permits["building_type"] = [building_type(text, sub_type) for text, sub_type in zip(permits.text, permits.permit_sub_type)]

steps = pd.Series({
    "applications downloaded": len(permits),
    "revisions removed": int(permits.is_revision.sum()),
    "plan-check-only filings removed": int((~permits.is_revision & permits.is_plan_check_only).sum()),
    "small structures removed": int((~permits.is_revision & ~permits.is_plan_check_only & permits.is_small_structure).sum()),
})
buildings = permits[~permits.is_revision & ~permits.is_plan_check_only & ~permits.is_small_structure].copy()
steps["real building applications kept"] = len(buildings)
print(steps.to_string())
print()
print(f"City data is current to {DATA_DATE.date()}")

## A trap in the data

Never assume a column was filled in on the day of filing. Below we compare how often each column has a value for applications that got a permit against applications that never did.

Some columns are almost always filled in for issued permits and often blank for the rest. The city completes them during review. A model that sees those columns learns "blank means no permit", which looks like great accuracy and is useless for a project that has not been reviewed yet. We keep only the columns that are filled in equally for both groups, and we read building details from the applicant's own description instead.

In [ ]:
checked_columns = ["use_desc", "square_footage", "height", "construction", "valuation", "zone", "apn", "lat", "work_desc"]
got_permit = buildings.permit_date.notna()
fill_rates = pd.DataFrame({
    "filled in, permit issued": buildings[got_permit][checked_columns].notna().mean(),
    "filled in, never issued": buildings[~got_permit][checked_columns].notna().mean(),
})
fill_rates["gap"] = fill_rates.iloc[:, 0] - fill_rates.iloc[:, 1]
fill_rates["safe to use"] = np.where(fill_rates.gap.abs() < 0.05, "yes", "no")
CITY_FILLED = list(fill_rates.index[fill_rates["safe to use"] == "no"])
fill_rates.round(2)

## From applications to projects

One development often files several applications on the same day, one per building. We group applications on the same site filed within 30 days of each other into one project. The clock starts at the first filing and stops at the first building permit. If the site had a plan-check-only filing in the four years before, the clock starts there. A few records show a permit on the very day they were filed, because the real review happened under an earlier record; when we cannot find that record we leave the project out.

Then we build the inputs for our models, using only what is known on filing day: construction cost, location, zoning, what the applicant wrote about the project (stories, units, floor area, affordable housing, underground parking and so on), the permit history of the site, and how fast the city has been moving lately.

In [ ]:
HISTORY_COLUMNS = ["prior_demo_3y", "prior_grading_3y", "prior_newbldg_5y", "prior_newbldg_issued", "n_prior_permits_5y"]
buildings = buildings.merge(site_history[["permit_nbr"] + HISTORY_COLUMNS], on="permit_nbr", how="left")
buildings[HISTORY_COLUMNS] = buildings[HISTORY_COLUMNS].fillna(0)


def site_of(table):
    street = table.primary_address.fillna("UNKNOWN").str.upper().str.replace(r"\s+\d[\d\s-]*$", "", regex=True)
    return table.apn.where(table.apn.notna(), "ADDRESS " + street)


buildings["site"] = site_of(buildings)
buildings = buildings.sort_values(["site", "filed"]).reset_index(drop=True)
starts_new_project = (buildings.site != buildings.site.shift()) | ((buildings.filed - buildings.filed.shift()).dt.days > 30)
buildings["project_id"] = starts_new_project.cumsum()

largest_first = buildings.sort_values("valuation", ascending=False).drop_duplicates("project_id").set_index("project_id")
grouped = buildings.groupby("project_id")
projects = grouped.agg(
    site=("site", "first"),
    filed=("filed", "min"),
    permit_date=("permit_date", "min"),
    occupancy_date=("occupancy_date", "min"),
    buildings_filed=("permit_nbr", "size"),
    valuation=("valuation", "sum"),
    lat=("lat", "median"),
    lon=("lon", "median"),
    text=("text", " | ".join),
    prior_demo_3y=("prior_demo_3y", "max"),
    prior_grading_3y=("prior_grading_3y", "max"),
    prior_newbldg_5y=("prior_newbldg_5y", "max"),
    prior_newbldg_issued=("prior_newbldg_issued", "max"),
    n_prior_permits_5y=("n_prior_permits_5y", "max"),
)
for column in ["permit_nbr", "primary_address", "zip_code", "building_type", "zone", "cd", "apc", "work_desc"] + CITY_FILLED:
    projects[column] = largest_first[column].reindex(projects.index)

early_reviews = permits[~permits.is_revision & permits.is_plan_check_only].copy()
early_reviews["site"] = site_of(early_reviews)
candidates = projects.reset_index()[["project_id", "site", "filed"]].merge(early_reviews[["site", "filed"]].rename(columns={"filed": "review_filed"}), on="site")
candidates = candidates[(candidates.review_filed < candidates.filed) & (candidates.review_filed >= candidates.filed - pd.Timedelta(days=4 * 365))]
first_review = candidates.groupby("project_id").review_filed.min().reindex(projects.index)

issued_on_filing_day = (projects.permit_date - projects.filed).dt.days <= 1
clock_unknown = issued_on_filing_day & first_review.isna()
projects["filed"] = first_review.fillna(projects.filed)
projects = projects[~clock_unknown].copy()

projects["days_to_permit"] = (projects.permit_date - projects.filed).dt.days.clip(lower=0)
projects["filing_year"] = projects.filed.dt.year
projects = projects.reset_index()

print(f"{len(buildings):,} applications grouped into {len(projects):,} projects")
print(f"{int(first_review.notna().sum())} projects had an earlier plan-check-only filing, so their clock starts there")
print(f"{int(clock_unknown.sum())} projects dropped: permit recorded on the filing day and no earlier filing found")
print(f"{projects.permit_date.notna().mean():.0%} of all projects have a permit so far")

In [ ]:
def first_number(text_column, pattern, largest):
    found = text_column.str.extract(pattern)[0].str.replace(",", "", regex=False)
    number = pd.to_numeric(found, errors="coerce")
    return number.where((number > 0) & (number <= largest))


projects["stories"] = first_number(projects.text, r"(\d+)[- ]?STOR", 80)
projects["units"] = first_number(projects.text, r"(\d+)[- ]?(?:UNITS?\b|DU\b|DWELLING|APARTMENTS?\b)", 3000)
projects["floor_area"] = first_number(projects.text, r"([\d,]{4,9}) ?(?:SQ\.? ?FT|SF\b|S\.F\.|SQUARE)", 5000000)
projects["valuation"] = projects.valuation.where(projects.valuation > 0)
projects["filing_month"] = projects.filed.dt.month

DESCRIPTION_FLAGS = {
    "affordable_housing": r"AFFORDABLE|\bVLI\b|\bELI\b|LOW INCOME",
    "transit_incentive": r"\bTOC\b|TRANSIT ORIENTED",
    "density_bonus": r"DENSITY BONUS|12\.22",
    "underground_parking": r"SUBTERRANEAN|BASEMENT|UNDERGROUND",
    "concrete_podium": r"PODIUM|TYPE I[- ]?A\b|TYPE 1[- ]?A\b",
    "wood_frame": r"TYPE V|TYPE III|TYPE 5|TYPE 3",
    "shell_only": r"SHELL",
    "electronic_plans": r"EPLAN|E-PLAN",
}
for name, pattern in DESCRIPTION_FLAGS.items():
    projects[name] = projects.text.str.contains(pattern, regex=True).astype(int)

zone_text = projects.zone.fillna("")
projects["zone_conditions"] = zone_text.str.contains(r"\(T\)|\(Q\)|\[Q\]|\[T\]", regex=True).astype(int)
projects["zone_overlay"] = zone_text.str.contains(r"CPIO|CDO|HPOZ|-SN|RIO|CUGU|POD|-O\b", regex=True).astype(int)
zone_base = zone_text.str.extract(r"^(?:\(T\)|\(Q\)|\[Q\]|\[T\])*([A-Z]+\d*(?:\.\d)?)")[0].fillna("UNKNOWN")
common_zones = zone_base.value_counts().index[:20]
projects["zone_base"] = zone_base.where(zone_base.isin(common_zones), "OTHER")
projects["council_district"] = projects.cd.fillna("UNKNOWN").astype(str)
projects["planning_area"] = projects.apc.fillna("UNKNOWN").astype(str)

projects[["filed", "building_type", "valuation", "stories", "units", "floor_area", "buildings_filed", "zone_base", "days_to_permit"]].head()

In [ ]:
def city_pace(projects):
    ordered = projects.sort_values("filed")
    filed_days = ordered.filed.values.astype("datetime64[D]").astype(int)
    quick = (ordered.days_to_permit <= 360).fillna(False).values.astype(float)
    running_quick = np.concatenate([[0.0], np.cumsum(quick)])
    window_start = np.searchsorted(filed_days, filed_days - 1080, side="left")
    window_end = np.searchsorted(filed_days, filed_days - 360, side="right")
    count = window_end - window_start
    recent_share = np.where(count >= 30, (running_quick[window_end] - running_quick[window_start]) / np.maximum(count, 1), np.nan)

    issued = projects[projects.permit_date.notna()].sort_values("permit_date")
    issue_days = issued.permit_date.values.astype("datetime64[D]").astype(int)
    waits = issued.days_to_permit.values
    year_start = np.searchsorted(issue_days, filed_days - 365, side="left")
    year_end = np.searchsorted(issue_days, filed_days, side="left")
    recent_wait = np.array([np.median(waits[a:b]) if b - a >= 30 else np.nan for a, b in zip(year_start, year_end)])
    return pd.DataFrame({"recent_quick_share": recent_share, "recent_typical_wait": recent_wait}, index=ordered.index)


projects[["recent_quick_share", "recent_typical_wait"]] = city_pace(projects)

NUMBERS = [
    "valuation", "stories", "units", "floor_area", "buildings_filed", "lat", "lon", "filing_month",
    "zone_conditions", "zone_overlay", "prior_demo_3y", "prior_grading_3y", "prior_newbldg_5y",
    "prior_newbldg_issued", "n_prior_permits_5y", "recent_quick_share", "recent_typical_wait",
] + list(DESCRIPTION_FLAGS)
CATEGORIES = ["building_type", "zone_base", "council_district", "planning_area"]
CATEGORY_TYPES = {name: pd.CategoricalDtype(sorted(projects[name].unique())) for name in CATEGORIES}

print(f"{len(NUMBERS) + len(CATEGORIES)} inputs per project, all known on filing day")

## A first look

Before any model: how often does a project have its permit two years after filing, and how long do the successful ones take? The second table shows the same thing by filing year. The city has been getting slower for a decade, which matters for every model below.

In [ ]:
def summarize(table, by):
    return table.groupby(by).agg(
        projects=("project_id", "size"),
        permit_within_12_months=("days_to_permit", lambda days: (days <= 360).mean()),
        permit_within_24_months=("days_to_permit", lambda days: (days <= 720).mean()),
        typical_days_when_issued=("days_to_permit", "median"),
    ).round(2)


two_years_old = projects[(DATA_DATE - projects.filed).dt.days >= 720]
by_class = summarize(two_years_old, "building_type")
by_year = summarize(two_years_old, "filing_year")
display(by_class)
display(by_year)

# Training and Testing

In our MAIA lessons we split students at random into 80% train and 20% test. That would be cheating here, because permits filed in the same year share the same conditions. We split by time instead: train on projects filed through 2018, using only what was known on January 1, 2019, and test on projects filed in 2019, 2020 and 2021. A project still waiting on the cutoff date counts as "still waiting", not as a failure. Every site stays on one side of the split.

In [ ]:
MONTHS = 48
LOG_SCALED = ["valuation", "stories", "units", "floor_area", "buildings_filed", "n_prior_permits_5y", "recent_typical_wait"]


def split_by_time(last_training_year, test_years):
    cutoff = pd.Timestamp(year=last_training_year + 1, month=1, day=1)
    test = projects[projects.filing_year.between(test_years[0], test_years[1])]
    train = projects[(projects.filed < cutoff) & ~projects.site.isin(set(test.site))]
    return train, test, cutoff


def as_known_on(table, cutoff):
    has_permit = (table.permit_date.notna() & (table.permit_date < cutoff)).values
    waited = np.where(has_permit, table.days_to_permit, (cutoff - table.filed).dt.days).astype(float)
    return np.clip(waited, 0, None), has_permit


def tree_table(table, extra=()):
    out = table[NUMBERS + list(extra)].astype(float).copy()
    for name in CATEGORIES:
        out[name] = table[name].astype(CATEGORY_TYPES[name])
    return out


def plain_table(table, fill_values):
    numbers = table[NUMBERS].astype(float).copy()
    for name in LOG_SCALED:
        numbers[name] = np.log1p(numbers[name])
    numbers = numbers.fillna(fill_values)
    labels = pd.get_dummies(pd.DataFrame({name: table[name].astype(CATEGORY_TYPES[name]) for name in ["building_type", "planning_area"]}), dtype=float)
    return pd.concat([numbers, labels], axis=1)


def fill_values_from(table):
    numbers = table[NUMBERS].astype(float).copy()
    for name in LOG_SCALED:
        numbers[name] = np.log1p(numbers[name])
    return numbers.median().fillna(0)


def score_days(actual, predicted):
    return {
        "RMSE (days)": np.sqrt(mean_squared_error(actual, predicted)),
        "typical miss (days)": np.median(np.abs(actual - predicted)),
        "R²": r2_score(actual, predicted),
    }


train, test, cutoff = split_by_time(2018, (2019, 2021))
train_waited, train_has_permit = as_known_on(train, cutoff)
finished_train = train[train_has_permit]
finished_test = test[test.days_to_permit <= MONTHS * 30]

print(f"Training projects: {len(train):,} ({len(finished_train):,} had a permit by {cutoff.date()})")
print(f"Test projects: {len(test):,} ({len(finished_test):,} have a permit so far)")

# Linear Regression

We start the way we did in our MAIA lessons: predict one number, the days from filing to permit, for projects that got a permit. The model learns on the logarithm of days, because a few projects take many years and would otherwise dominate.

To judge it we need something simple to beat. Our baseline is the typical wait for that building type over the three years before the cutoff.

Two scores, the same ones we used in the lessons:

- **RMSE** (root mean squared error): roughly how far off the predictions are, in days. Big misses count extra. Lower is better.
- **R²**: the share of the differences between projects that the model explains. 1 is perfect, 0 is no better than guessing the average wait for every project, and below 0 is worse than that.

In [ ]:
recent_finished = finished_train[finished_train.permit_date >= cutoff - pd.Timedelta(days=3 * 365)]
typical_by_class = recent_finished.groupby("building_type").days_to_permit.median()
baseline_days = finished_test.building_type.map(typical_by_class).fillna(recent_finished.days_to_permit.median()).values

fill_values = fill_values_from(finished_train)
X_train = plain_table(finished_train, fill_values)
X_test = plain_table(finished_test, fill_values)
y_train = np.log1p(finished_train.days_to_permit)
y_test = finished_test.days_to_permit.values

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_days = np.expm1(linear_model.predict(X_test)).clip(0, MONTHS * 30)

day_scores = {"Baseline (typical wait)": score_days(y_test, baseline_days), "Linear Regression": score_days(y_test, linear_days)}
print("Baseline RMSE (days):", round(day_scores["Baseline (typical wait)"]["RMSE (days)"]))
print("Linear Regression RMSE (days):", round(day_scores["Linear Regression"]["RMSE (days)"]))
print("Linear Regression R² Score:", round(day_scores["Linear Regression"]["R²"], 3))

# Neural Network Regression

Can a neural network find patterns the straight line missed? Same inputs, same split. Neural networks are sensitive to the size of the numbers they are fed, so we scale every input first.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

nn_model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.01,
    early_stopping=True,
    max_iter=2000,
    random_state=67,
)
nn_model.fit(X_train_scaled, y_train)
nn_days = np.expm1(nn_model.predict(X_test_scaled)).clip(0, MONTHS * 30)

day_scores["Neural Network"] = score_days(y_test, nn_days)
print("Neural Network RMSE (days):", round(day_scores["Neural Network"]["RMSE (days)"]))
print("Neural Network R² Score:", round(day_scores["Neural Network"]["R²"], 3))

# Gradient Boosting

Gradient boosting builds hundreds of small decision trees, each one correcting the mistakes of the ones before it. It is usually the strongest choice for table data like ours, and it handles missing values and categories without extra work.

In [ ]:
boosting_settings = dict(
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=40,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_lambda=5,
    random_state=67,
    verbose=-1,
)
boosting_model = lgb.LGBMRegressor(**boosting_settings)
boosting_model.fit(tree_table(finished_train), y_train)
boosting_days = np.expm1(boosting_model.predict(tree_table(finished_test))).clip(0, MONTHS * 30)

day_scores["Gradient Boosting"] = score_days(y_test, boosting_days)
day_results = pd.DataFrame(day_scores).T
day_results.round({"RMSE (days)": 0, "typical miss (days)": 0, "R²": 3})

# From One Number to Odds

All three models above miss by many months, none beats the simple baseline, and every R² is below zero. Two things went wrong, and neither is the model's fault:

1. **They only learn from finished projects.** On the cutoff date, the recently filed projects that already have a permit are the fast ones. The slow ones are still waiting and get left out, so the models learn that recent projects are quick.
2. **A single number hides the real risk.** Many projects never get a permit at all, and "days to permit" does not exist for them.

So we change the question. For every month a project is still waiting, our model estimates the chance that its permit arrives that month. Chaining the months together gives the chance of having a permit by any date. A project that is still waiting on the cutoff date teaches the model something too: it has not been issued yet. If we have only seen part of its latest month, that month counts as a part month.

We compare three ways of giving odds:

- **Track record:** no machine learning. For each month of waiting, what share of projects of this building type got their permit that month, counting only the last two years of city activity.
- **Odds model:** gradient boosting on the monthly question above, using everything we know about the project.
- **Blend:** the average of the two.

Two scores, both on projects the model never saw:

- **Ranking score:** if you pick one project that got its permit in time and one that did not, how often did the method give the first one better odds? 0.5 is a coin flip, 1.0 is perfect.
- **Odds error:** the average squared gap between the odds given and what happened. Lower is better.

A month here means 30 days, so "within 12 months" is 360 days and "within 24 months" is 720.

In [ ]:
odds_settings = dict(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=80,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_lambda=5,
    random_state=67,
    verbose=-1,
)


def waiting_months(table, waited, has_permit):
    last_month = np.minimum(waited // 30, MONTHS - 1).astype(int)
    counts = last_month + 1
    project = np.repeat(np.arange(len(table)), counts)
    months_waited = np.concatenate([np.arange(count) for count in counts])
    is_last_month = months_waited == np.repeat(last_month, counts)
    arrived = has_permit & (waited < MONTHS * 30)
    permit_came = (is_last_month & np.repeat(arrived, counts)).astype(int)
    part_of_month_seen = np.where(has_permit | (waited >= MONTHS * 30), 1.0, (waited % 30) / 30)
    exposure = np.where(is_last_month, np.repeat(part_of_month_seen, counts), 1.0)
    calendar_date = table.filed.values[project] + (months_waited * 30).astype("timedelta64[D]")
    return project, months_waited, permit_came, calendar_date, exposure


def fit_odds_model(table, waited, has_permit, extra=()):
    project, months_waited, permit_came, calendar_date, exposure = waiting_months(table, waited, has_permit)
    rows = tree_table(table, extra).iloc[project].reset_index(drop=True)
    rows["months_waited"] = months_waited
    model = lgb.LGBMClassifier(**odds_settings)
    model.fit(rows, permit_came, sample_weight=np.maximum(exposure, 0.000001))
    return model


def permit_curves(model, table, extra=()):
    features = tree_table(table, extra)
    rows = features.iloc[np.repeat(np.arange(len(features)), MONTHS)].reset_index(drop=True)
    rows["months_waited"] = np.tile(np.arange(MONTHS), len(features))
    monthly_chance = model.predict_proba(rows)[:, 1].reshape(len(features), MONTHS)
    return 1 - np.cumprod(1 - monthly_chance, axis=1)


def track_record(table, waited, has_permit, cutoff):
    project, months_waited, permit_came, calendar_date, exposure = waiting_months(table, waited, has_permit)
    last_two_years = calendar_date >= np.datetime64(cutoff - pd.Timedelta(days=730))
    building_type = table.building_type.values[project]

    def monthly_chance(chosen):
        waiting = np.bincount(months_waited[chosen], weights=exposure[chosen], minlength=MONTHS)
        permits = np.bincount(months_waited[chosen], weights=permit_came[chosen], minlength=MONTHS)
        return np.minimum(permits / np.maximum(waiting, 1), 1), waiting

    all_types, waiting = monthly_chance(last_two_years)
    all_types = pd.Series(np.where(waiting >= 5, all_types, np.nan)).ffill().fillna(0).values
    curves = {}
    for name in CATEGORY_TYPES["building_type"].categories:
        own, waiting = monthly_chance(last_two_years & (building_type == name))
        trust = np.minimum(waiting / 150, 1)
        curves[name] = 1 - np.cumprod(1 - (trust * own + (1 - trust) * all_types))
    return curves


def track_record_curves(table, waited, has_permit, cutoff, classes):
    curves = track_record(table, waited, has_permit, cutoff)
    return np.vstack([curves[name] for name in classes])


def score_odds(table, curves):
    rows = []
    for months in (12, 24, 36):
        watched_long_enough = ((DATA_DATE - table.filed).dt.days >= months * 30).values
        actual = (table.days_to_permit <= months * 30).values[watched_long_enough]
        for name, curve in curves.items():
            chance = curve[watched_long_enough, months - 1]
            rows.append({
                "permit within": f"{months} months",
                "method": name,
                "projects": int(watched_long_enough.sum()),
                "actual share": actual.mean(),
                "average odds given": chance.mean(),
                "ranking score": roc_auc_score(actual, chance),
                "odds error": np.mean((chance - actual) ** 2),
            })
    return pd.DataFrame(rows).set_index(["permit within", "method"]).round(3)


def month_reached(curves, share):
    share_of_final = curves / np.maximum(curves[:, -1:], 1e-9)
    return np.argmax(share_of_final >= share, axis=1) + 1


def typical_days(curves):
    return month_reached(curves, 0.5) * 30 - 15


def range_holds(table, curves, low, high):
    watched_four_years = ((DATA_DATE - table.filed).dt.days >= MONTHS * 30).values
    finished = (table.days_to_permit <= MONTHS * 30).values & watched_four_years
    actual = table.days_to_permit.values[finished]
    earliest = (month_reached(curves, low)[finished] - 1) * 30
    latest = month_reached(curves, high)[finished] * 30
    return np.mean((actual >= earliest) & (actual <= latest))


def backtest(last_training_year, test_years, extra=()):
    train, test, cutoff = split_by_time(last_training_year, test_years)
    waited, has_permit = as_known_on(train, cutoff)
    model = fit_odds_model(train, waited, has_permit, extra)
    curves = {
        "Track record": track_record_curves(train, waited, has_permit, cutoff, test.building_type),
        "Odds model": permit_curves(model, test, extra),
    }
    curves["Blend"] = (curves["Track record"] + curves["Odds model"]) / 2
    return test, curves, model

In [ ]:
test, curves, odds_model = backtest(2018, (2019, 2021))
odds_results = score_odds(test, curves)

finished = (test.days_to_permit <= MONTHS * 30).values
odds_days = typical_days(curves["Odds model"])[finished]
day_scores["Odds model (middle estimate)"] = score_days(y_test, odds_days)
day_results = pd.DataFrame(day_scores).T

print("Projects filed 2019 to 2021, trained on what was known on January 1, 2019")
display(odds_results)

One test period can be luck. We repeat the whole exercise for four separate periods, each time training only on what was known before the test projects were filed. The report card below shows, for each period:

- the odds error of the track record and of the blend for "permit within 24 months" (lower is better),
- the ranking score of each,
- how often the real wait fell inside the blend's date ranges. The **80% range** runs from the month by which the fastest 10% of permits should have arrived to the month by which 90% should have. The **50% range** runs from the 25% mark to the 75% mark; these are the "optimistic" and "pessimistic" months the tool prints. A range built to hold 80% of projects should hold about 80% of them, and the 50% range about 50%. The ranges describe projects that do get a permit, so we check them on test projects that got one within four years and that we have been able to watch for four years. The chance of getting no permit at all is a separate number: it is what the odds measure.

In [ ]:
periods = {"2013 to 2015": (2012, (2013, 2015)), "2016 to 2018": (2015, (2016, 2018)), "2019 to 2021": (2018, (2019, 2021)), "2022 to 2023": (2021, (2022, 2023))}
backtests = {"2019 to 2021": (test, curves)}
report_rows = []
for name, (last_training_year, test_years) in periods.items():
    if name not in backtests:
        period_test, period_curves, period_model = backtest(last_training_year, test_years)
        backtests[name] = (period_test, period_curves)
    period_test, period_curves = backtests[name]
    scores = score_odds(period_test, period_curves).loc["24 months"]
    report_rows.append({
        "projects filed": name,
        "projects": len(period_test),
        "permit within 24 months": scores.loc["Blend", "actual share"],
        "odds error, track record": scores.loc["Track record", "odds error"],
        "odds error, blend": scores.loc["Blend", "odds error"],
        "ranking, track record": scores.loc["Track record", "ranking score"],
        "ranking, blend": scores.loc["Blend", "ranking score"],
        "80% range held": range_holds(period_test, period_curves["Blend"], 0.1, 0.9),
        "50% range held": range_holds(period_test, period_curves["Blend"], 0.25, 0.75),
    })
report_card = pd.DataFrame(report_rows).set_index("projects filed").round(3)
report_card

# Catching a Model That Cheats

Remember the columns the city fills in during review: building use, floor area, height and construction type. What happens if we hand them to the same odds model?

Its ranking score jumps. It looks like a much better model. It is not. We call it the cheating model: it has learned that a blank floor area means the city never finished reviewing the project. On filing day every project has those columns blank, so the extra "accuracy" disappears the moment the tool is used for real.

In [ ]:
projects["city_floor_area"] = projects.square_footage
projects["city_height"] = projects.height
projects["city_use_filled"] = projects.use_desc.notna().astype(int)
projects["city_construction_filled"] = projects.construction.notna().astype(int)
CHEATING_INPUTS = ["city_floor_area", "city_height", "city_use_filled", "city_construction_filled"]

cheat_test, cheat_curves, cheat_model = backtest(2018, (2019, 2021), extra=CHEATING_INPUTS)
cheat_results = score_odds(cheat_test, {"Cheating model": cheat_curves["Odds model"]})

honest_scores = odds_results.xs("Odds model", level="method")["ranking score"]
cheating_scores = cheat_results.xs("Cheating model", level="method")["ranking score"]
leak_comparison = pd.DataFrame({"odds model, honest inputs": honest_scores, "cheating model": cheating_scores})
leak_comparison["fake gain"] = leak_comparison.iloc[:, 1] - leak_comparison.iloc[:, 0]
leak_comparison

# Comparing Results

Let's take one real project from the test set and see what each model said about it on filing day, next to what actually happened.

In [ ]:
position = 40
example = finished_test.iloc[[position]]
example_row = np.flatnonzero(test.project_id.values == example.project_id.values[0])[0]

print("--- Project ---")
display(example[["filed", "building_type", "valuation", "stories", "units", "primary_address"]])
print(f"Description: {example.work_desc.values[0][:200]}")
print()
print(f"Actual days to permit: {y_test[position]:.0f}")
print(f"Baseline (typical wait): {baseline_days[position]:.0f}")
print(f"Linear Regression: {linear_days[position]:.0f}")
print(f"Neural Network: {nn_days[position]:.0f}")
print(f"Gradient Boosting: {boosting_days[position]:.0f}")
print(f"Odds model (middle estimate): {odds_days[position]:.0f}")
print()
for months in (12, 24, 36):
    happened = "yes" if y_test[position] <= months * 30 else "no"
    print(f"Odds of a permit within {months} months (Blend): {curves['Blend'][example_row, months - 1]:.0%}   Did it happen? {happened}")

# Implementing Our Model

For the tool itself we retrain on every project through today, including the ones still waiting, and rebuild the track record from the last two years. The estimate you get is the blend of the two.

Fill in the form and press **Estimate**. Leave a number at 0 if you do not know it. You get:

- the odds of having the building permit within 12, 24 and 36 months,
- an optimistic, middle and pessimistic month for when the permit comes, if it comes (the middle estimate is the month by which half of the permits that do come have arrived),
- how each box you ticked moves the odds,
- the most similar past projects and what happened to them,
- typical times for the planning approval before filing (if your project needs one) and for construction after the permit. These two are simple lookups from city records, not model predictions.

In [ ]:
all_waited, all_have_permit = as_known_on(projects, DATA_DATE + pd.Timedelta(days=1))
final_model = fit_odds_model(projects, all_waited, all_have_permit)
todays_track_record = track_record(projects, all_waited, all_have_permit, DATA_DATE)

newest_first = projects.sort_values("filed", ascending=False)
todays_pace = {name: newest_first[name].dropna().iloc[0] for name in ["recent_quick_share", "recent_typical_wait"]}
typical_site_permits = projects.n_prior_permits_5y.median()
uses_electronic_plans = int(newest_first.electronic_plans.head(300).mean() > 0.5)


def most_common(values):
    return values.value_counts().index[0]


projects["zip5"] = projects.zip_code.fillna("").astype(str).str[:5]
zip_table = projects[projects.zip5.str.len() == 5].groupby("zip5").agg(
    lat=("lat", "median"),
    lon=("lon", "median"),
    council_district=("council_district", most_common),
    planning_area=("planning_area", most_common),
    zone_base=("zone_base", most_common),
)
citywide = {"lat": projects.lat.median(), "lon": projects.lon.median(), "council_district": most_common(projects.council_district), "planning_area": most_common(projects.planning_area), "zone_base": most_common(projects.zone_base)}


def describe_project(building_type, valuation, stories, units, floor_area, buildings_filed, zip_code, zone, ticked):
    place = zip_table.loc[zip_code].to_dict() if zip_code in zip_table.index else dict(citywide)
    row = {name: 0.0 for name in NUMBERS}
    row.update({
        "valuation": valuation if valuation > 0 else np.nan,
        "stories": stories if stories > 0 else np.nan,
        "units": units if units > 0 else np.nan,
        "floor_area": floor_area if floor_area > 0 else np.nan,
        "buildings_filed": max(buildings_filed, 1),
        "lat": place["lat"],
        "lon": place["lon"],
        "filing_month": DATA_DATE.month,
        "n_prior_permits_5y": typical_site_permits,
        "electronic_plans": uses_electronic_plans,
        "recent_quick_share": todays_pace["recent_quick_share"],
        "recent_typical_wait": todays_pace["recent_typical_wait"],
        "building_type": building_type,
        "zone_base": place["zone_base"] if zone == "Typical for the ZIP code" else zone,
        "council_district": place["council_district"],
        "planning_area": place["planning_area"],
    })
    for name in ticked:
        row[name] = 1.0
    return pd.DataFrame([row])


def blended_curve(row):
    model_curve = permit_curves(final_model, row)[0]
    record_curve = todays_track_record[row.building_type.iloc[0]]
    return (model_curve + record_curve) / 2, model_curve, record_curve


def similar_projects(row, how_many=25):
    pool = projects[(projects.building_type == row.building_type.iloc[0]) & (projects.filed >= DATA_DATE - pd.Timedelta(days=8 * 365)) & projects.valuation.notna()]
    target_value = row.valuation.iloc[0] if pd.notna(row.valuation.iloc[0]) else pool.valuation.median()
    distance = (np.log(pool.valuation) - np.log(target_value)).abs()
    for name in ["units", "stories"]:
        if pd.notna(row[name].iloc[0]):
            gap = (np.log1p(pool[name]) - np.log1p(row[name].iloc[0])).abs()
            distance = distance + 0.5 * gap.fillna(gap.median())
    return pool.loc[distance.sort_values().index[:how_many]]


print(f"Final model trained on {len(projects):,} projects through {DATA_DATE.date()}")

In [ ]:
NO_APPROVAL = "None (the project fits the zoning as it is)"
TICK_BOXES = {
    "affordable_housing": "Includes affordable housing",
    "transit_incentive": "Uses the transit incentive program (TOC: extra density near transit)",
    "density_bonus": "Uses a density bonus (extra units in return for affordable ones)",
    "underground_parking": "Underground parking",
    "concrete_podium": "Concrete podium or Type I (concrete or steel) construction",
    "wood_frame": "Wood frame (Type III or V)",
    "shell_only": "Shell and core only (interiors built out later)",
    "zone_conditions": "Zoning carries Q or T conditions (site-specific limits)",
    "zone_overlay": "Site is in an overlay district (extra design rules)",
    "prior_demo_3y": "Demolition permit on the site in the last 3 years",
    "prior_grading_3y": "Grading permit on the site in the last 3 years",
    "prior_newbldg_5y": "Earlier new-building application on the site",
}


def month_label(months):
    return f"month {months} (about {(DATA_DATE + pd.DateOffset(months=int(months))).strftime('%b %Y')})"


def print_estimate(building_type, valuation, stories, units, floor_area, buildings_filed, zip_code, zone, ticked, planning_approval):
    row = describe_project(building_type, valuation, stories, units, floor_area, buildings_filed, zip_code, zone, ticked)
    blend, model_curve, record_curve = blended_curve(row)
    if zip_code not in zip_table.index:
        print(f"ZIP code {zip_code} has no past projects in our data, so a typical Los Angeles location is used.")
        print()

    print(f"NEW {building_type.upper()} PROJECT FILED TODAY")
    print()
    print("Chance of having the building permit")
    for months in (12, 24, 36):
        print(f"   within {months} months: {blend[months - 1]:.0%}")
    print(f"   within 4 years:   {blend[-1]:.0%}")
    print()
    print("If the permit comes, when")
    labels = {0.25: "optimistic (1 in 4 are faster)", 0.5: "middle estimate", 0.75: "pessimistic (1 in 4 are slower)"}
    for share, label in labels.items():
        print(f"   {label}: {month_label(month_reached(blend[None, :], share)[0])}")
    print()
    print("Where the 24-month number comes from")
    print(f"   track record for {building_type}, last two years: {record_curve[23]:.0%}")
    print(f"   odds model for this project: {model_curve[23]:.0%}")

    if ticked:
        print()
        print("How each ticked box moves the 24-month odds")
        for name in ticked:
            without = row.copy()
            without[name] = 0.0
            change = (blend[23] - blended_curve(without)[0][23]) * 100
            print(f"   {TICK_BOXES[name]}: {change:+.0f} {'point' if abs(round(change)) == 1 else 'points'}")

    neighbors = similar_projects(row)
    old_enough = neighbors[(DATA_DATE - neighbors.filed).dt.days >= 720]
    with_permit = neighbors[neighbors.days_to_permit.notna()]
    print()
    print(f"The {len(neighbors)} most similar {building_type} projects filed in the last 8 years")
    print(f"   {int((old_enough.days_to_permit <= 720).sum())} of the {len(old_enough)} filed at least two years ago had a permit within 24 months")
    if len(with_permit):
        print(f"   those with a permit waited {with_permit.days_to_permit.quantile(0.25):.0f} to {with_permit.days_to_permit.quantile(0.75):.0f} days (middle half), typically {with_permit.days_to_permit.median():.0f}")
    closest = neighbors.head(5)[["filed", "primary_address", "valuation", "stories", "units", "days_to_permit"]].copy()
    closest["filed"] = closest.filed.dt.date
    closest["days_to_permit"] = closest.days_to_permit.map(lambda days: "still waiting" if pd.isna(days) else f"{days:.0f}")
    display(closest)

    if planning_approval != NO_APPROVAL:
        approval = entitlement_times[entitlement_times.plain_name == planning_approval].iloc[0]
        print()
        print(f"Before the permit: {planning_approval}")
        print(f"   typically {approval.median_days / 30:.0f} months; the middle half of cases took {approval.p25_days / 30:.0f} to {approval.p75_days / 30:.0f} months ({approval.n_cases:.0f} completed cases, 2020 to 2025)")
        print("   plan check can overlap with it, so do not simply add the two")
    build = construction_times[construction_times.building_type == building_type].iloc[0]
    print()
    print("After the permit: construction")
    print(f"   typically {build.median_months:.0f} months from permit to certificate of occupancy; the middle half took {build.p25_months:.0f} to {build.p75_months:.0f} months")

    months = np.arange(1, MONTHS + 1)
    plt.figure(figsize=(8, 4))
    plt.plot(months, blend * 100, color="#1f77b4", linewidth=2.5, label="This project (blend)")
    plt.plot(months, record_curve * 100, color="gray", linestyle="--", label=f"Track record, {building_type}")
    plt.xlabel("Months after filing")
    plt.ylabel("Chance of having the permit (%)")
    plt.title("Chance of having the building permit by each month")
    plt.ylim(0, 100)
    plt.legend()
    plt.show()

In [ ]:
wide = {"description_width": "230px"}
box_layout = widgets.Layout(width="460px")
class_choice = widgets.Dropdown(options=list(CATEGORY_TYPES["building_type"].categories), value="multifamily", description="Building type", style=wide, layout=box_layout)
valuation_box = widgets.IntText(value=15000000, description="Construction cost ($)", style=wide, layout=box_layout)
stories_box = widgets.IntText(value=5, description="Stories", style=wide, layout=box_layout)
units_box = widgets.IntText(value=60, description="Housing units (0 if none)", style=wide, layout=box_layout)
area_box = widgets.IntText(value=60000, description="Floor area (square feet)", style=wide, layout=box_layout)
buildings_box = widgets.IntText(value=1, description="Buildings in the project", style=wide, layout=box_layout)
zip_box = widgets.Text(value="90015", description="ZIP code", style=wide, layout=box_layout)
zone_choice = widgets.Dropdown(options=["Typical for the ZIP code"] + list(CATEGORY_TYPES["zone_base"].categories), description="Zoning", style=wide, layout=box_layout)
approval_choice = widgets.Dropdown(options=[NO_APPROVAL] + list(entitlement_times.plain_name), description="Planning approval needed", style=wide, layout=box_layout)
tick_boxes = {name: widgets.Checkbox(value=False, description=label, indent=False, layout=widgets.Layout(width="640px")) for name, label in TICK_BOXES.items()}
estimate_button = widgets.Button(description="Estimate", button_style="primary")
result_area = widgets.Output()


def on_estimate(button):
    ticked = [name for name, box in tick_boxes.items() if box.value]
    result_area.clear_output()
    with result_area:
        print_estimate(
            class_choice.value, valuation_box.value, stories_box.value, units_box.value, area_box.value,
            buildings_box.value, zip_box.value.strip()[:5], zone_choice.value, ticked, approval_choice.value,
        )


estimate_button.on_click(on_estimate)
display(widgets.VBox(
    [class_choice, valuation_box, stories_box, units_box, area_box, buildings_box, zip_box, zone_choice, approval_choice]
    + list(tick_boxes.values()) + [estimate_button, result_area]
))

In [ ]:
print_estimate(
    "multifamily", 15000000, 5, 60, 60000, 1, "90015", "Typical for the ZIP code",
    ["affordable_housing", "underground_parking", "wood_frame"], NO_APPROVAL,
)

# Visualizations

## Predicting one number: days to permit

In [ ]:
short_names = ["Baseline\n(typical wait)", "Linear\nRegression", "Neural\nNetwork", "Gradient\nBoosting", "Odds model\n(middle estimate)"]
bar_colors = ["gray", "#9ecae1", "#6baed6", "#3182bd", "#2ca02c"]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
rmse_values = day_results["RMSE (days)"].values
axes[0].bar(short_names, rmse_values, color=bar_colors)
axes[0].set_title("Root Mean Squared Error in days (lower is better)")
axes[0].set_ylabel("RMSE (days)")
axes[0].set_ylim(0, rmse_values.max() * 1.12)
for i, value in enumerate(rmse_values):
    axes[0].text(i, value + 8, f"{value:.0f}", ha="center", fontweight="bold")

r2_values = day_results["R²"].values
axes[1].bar(short_names, r2_values, color=bar_colors)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("R² score (higher is better; below 0 is worse than guessing the average)")
axes[1].set_ylabel("R² score")
axes[1].set_ylim(r2_values.min() - 0.3, 0.4)
for i, value in enumerate(r2_values):
    axes[1].text(i, value - 0.12 if value < 0 else value + 0.04, f"{value:.2f}", ha="center", fontweight="bold")

plt.suptitle("Days to permit, projects filed 2019 to 2021")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.scatterplot(x=y_test, y=odds_days, alpha=0.4, ax=axes[0], color="teal")
ideal_line = np.linspace(0, MONTHS * 30, 100)
axes[0].plot(ideal_line, ideal_line, color="red", linestyle="--", label="Perfect prediction")
axes[0].set_title("Actual vs. predicted days to permit (odds model, middle estimate)")
axes[0].set_xlabel("Actual days to permit")
axes[0].set_ylabel("Predicted days to permit")
axes[0].legend()

residuals = y_test - odds_days
sns.histplot(residuals, kde=True, ax=axes[1], color="purple")
axes[1].axvline(x=0, color="red", linestyle="--")
axes[1].set_title("Distribution of prediction errors")
axes[1].set_xlabel("Actual minus predicted (days)")
axes[1].set_ylabel("Projects")

plt.tight_layout()
plt.show()

The scatter is a cloud, not a line: none of our models could tell a 300-day project from a 900-day project on filing day.

## Predicting odds, and the model that cheats

In [ ]:
at_24_months = odds_results.loc["24 months"]
ranking = [at_24_months.loc["Track record", "ranking score"], at_24_months.loc["Odds model", "ranking score"], at_24_months.loc["Blend", "ranking score"], cheat_results.loc[("24 months", "Cheating model"), "ranking score"]]
ranking_names = ["Track record", "Odds model", "Blend", "Cheating model"]

groups = []
for name, (period_test, period_curves) in backtests.items():
    chance = period_curves["Blend"][:, 23]
    happened = (period_test.days_to_permit <= 720).values
    groups.append(pd.DataFrame({"odds given": chance, "happened": happened}))
pooled = pd.concat(groups)
pooled["group"] = pd.qcut(pooled["odds given"], 8)
calibration = pooled.groupby("group", observed=True).agg(given=("odds given", "mean"), happened=("happened", "mean"))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
axes[0].bar(ranking_names, ranking, color=["gray", "#3182bd", "#2ca02c", "#d62728"])
axes[0].axhline(0.5, color="black", linestyle=":", label="Coin flip")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Ranking score")
axes[0].set_title("Permit within 24 months: who ranks projects best?\n(the red bar is not real skill)")
for i, value in enumerate(ranking):
    axes[0].text(i, value + 0.01, f"{value:.2f}", ha="center", fontweight="bold")
axes[0].legend()

axes[1].plot([0, 1], [0, 1], color="red", linestyle="--", label="Perfectly honest odds")
axes[1].plot(calibration.given, calibration.happened, marker="o", color="#2ca02c", linewidth=2, label="Blend, four test periods (each dot is one eighth of the projects)")
axes[1].set_xlabel("Odds the tool gave for a permit within 24 months")
axes[1].set_ylabel("Share of those projects that got one")
axes[1].set_title("Do the odds come true?")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.tight_layout()
plt.show()

## The city has been getting slower, and it is not only Los Angeles

We ran the same measurement on Seattle's open permit data. The share of new commercial and multifamily applications that had a permit within 24 months fell in both cities. Seattle started much higher, fell through 2021 and has partly recovered since; Los Angeles fell and has stayed low. This chart counts every application separately, including each building of a multi-building project, so the Los Angeles line sits a little above the project figures earlier in the notebook. This is why the odds in the chart above sit below the red line: a tool trained on the past is always a little optimistic when the future is slower.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for city, color in [("Los Angeles", "#1f77b4"), ("Seattle", "#ff7f0e")]:
    rows = city_benchmark[city_benchmark.city == city]
    axes[0].plot(rows.filing_year, rows.share_24m * 100, marker="o", linewidth=2, color=color, label=city)
axes[0].set_title("Share of applications with a permit within 24 months")
axes[0].set_xlabel("Year the application was filed")
axes[0].set_ylabel("Share with a permit (%)")
axes[0].set_ylim(0, 100)
axes[0].legend()

months = np.arange(1, MONTHS + 1)
for name, curve in todays_track_record.items():
    axes[1].plot(months, curve * 100, linewidth=2, label=name)
axes[1].set_title("Los Angeles track record today, by building type")
axes[1].set_xlabel("Months after filing")
axes[1].set_ylabel("Share with a permit (%)")
axes[1].set_ylim(0, 100)
axes[1].legend()

plt.tight_layout()
plt.show()

## What moves the odds

For recent projects we switch each yes/no input on and then off and record how the model's 24-month odds change on average. These are patterns in past projects, not causes: a project with underground parking is probably slower because it tends to be a bigger, more complex building, not because of the parking itself.

In [ ]:
recent_projects = projects[projects.filed >= DATA_DATE - pd.Timedelta(days=5 * 365)]
effects = {}
for name, label in TICK_BOXES.items():
    switched_on = recent_projects.copy()
    switched_on[name] = 1
    switched_off = recent_projects.copy()
    switched_off[name] = 0
    effects[label] = (permit_curves(final_model, switched_on)[:, 23] - permit_curves(final_model, switched_off)[:, 23]).mean() * 100
effects = pd.Series(effects).sort_values()

plt.figure(figsize=(10, 6))
plt.barh(effects.index, effects.values, color=np.where(effects.values < 0, "#d62728", "#2ca02c"))
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("Change in 24-month permit odds (percentage points)")
plt.title("How each project feature moves the odds")
plt.tight_layout()
plt.show()

# What We Learned

The numbers below are from city data through mid-November 2026. Rerunning the notebook later will move them slightly.

**Key findings**

1. **Getting a permit for a new building in Los Angeles is slow and far from certain.** Of projects filed in 2022 and 2023, about 28% had a building permit two years later. For projects filed in 2013 to 2015 it was about 70%.
2. **Predicting a single date does not work.** Linear regression, a neural network and gradient boosting all had an RMSE above 14 months (440 to 525 days) and lost to the baseline, the plain typical wait for that building type (383 days). Their R² scores are below zero. Only the middle estimate of our odds model beat the baseline (325 days), and its R² is still close to zero.
3. **Odds and ranges do work.** Across four separate test periods, for projects that got a permit, the real wait fell inside the tool's 80% range between 77% and 82% of the time, and inside its 50% range (optimistic to pessimistic) between 48% and 52% of the time. The blend ranks projects better than the track record alone (ranking score of 0.57 to 0.65 against 0.50 to 0.58, where 0.5 is a coin flip) and has lower odds error in two of four periods (5% and 11% lower), with a tie in the other two.
4. **A model can cheat without anyone noticing.** Giving the odds model four columns that the city fills in during review (the cheating model) raised its ranking score for a 24-month permit from 0.64 to 0.74. None of that gain is real.
5. **The slowdown is not unique to Los Angeles.** Seattle's permit rate also fell over the same years, from a much higher starting point, and has partly recovered since 2021. Los Angeles has stayed low.

**Limitations**

- **The odds lean optimistic when the city keeps slowing.** For projects filed 2019 to 2021 the blend gave an average 54% chance of a permit within 24 months and 37% got one. The ranges held up better than the headline odds.
- **Most of what decides the wait is not in public data:** how fast the applicant answers corrections, financing, redesigns. The ranking scores of 0.6 to 0.65 are modest for that reason, and we tested several other data sources (City Planning cases, the filer's track record, county assessor records, a wider set of permits) that did not change this.
- **Building type comes from the applicant's description.** A hand check of 300 descriptions found our first set of rules right about 77% of the time; we fixed the rules it flagged, but some descriptions stay ambiguous.
- **Plan-check-only filings are left out** unless a later permit application on the same site connects to them. Projects whose parcel number changed along the way can be missed. About 100 projects whose permit is recorded on the filing day, with no earlier filing we could find, are also left out, which makes the permit rates here slightly low.
- **Hotels are rare** (about 50 projects), so hotel estimates lean on the record for all building types.
- **The planning approval and construction times are simple lookups** from completed cases and finished buildings. They leave out cases that are still open, so they run a little short.